![image_1788772894693.png](./image_1788772894693.png "image_1788772894693.png")

#### Experiment: Trigger Types

Testing `once` vs `availableNow` vs `processingTime` on the same Auto Loader source,
using the same checkpoint (so each trigger only sees newly arrived files).

In [0]:
from pyspark.sql.functions import col

CATALOG = "dbr_dev_ua5816bd"
SCHEMA_LANDING = "roksolana_shendiu770"
SCHEMA_BRONZE = "roksolana_shendiu770_bronze"

SOURCE_PATH = f"/Volumes/{CATALOG}/{SCHEMA_LANDING}/bronze_landing/petroleum_consumption"
SCHEMA_LOCATION = f"/Volumes/{CATALOG}/{SCHEMA_LANDING}/bronze_landing/_schemas/petroleum_consumption"
CHECKPOINT_LOCATION = f"/Volumes/{CATALOG}/{SCHEMA_LANDING}/bronze_landing/_checkpoints/petroleum_consumption"
TARGET_TABLE = f"{CATALOG}.{SCHEMA_BRONZE}.petroleum_consumption_bronze"

raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumnsWithTypeWidening")
    .option("cloudFiles.maxFilesPerTrigger", 100)
    .load(SOURCE_PATH)
    .select("*", col("_metadata.file_path").alias("source_file_path"))
)

query_once = (
    raw_stream.writeStream
    .option("checkpointLocation", CHECKPOINT_LOCATION)
    .option("mergeSchema", "true")
    .trigger(once=True)
    .toTable(TARGET_TABLE)
)

query_once.awaitTermination()

In [0]:
%sql
DESCRIBE HISTORY dbr_dev_ua5816bd.roksolana_shendiu770_bronze.petroleum_consumption_bronze

**Trigger: `once=True`** – processed all 15 new files in a single micro-batch
(409 rows, 1 commit), ignoring `maxFilesPerTrigger`. Unlike `availableNow`, which
respects rate-limiting and splits work into multiple micro-batches, `once` has no
such control.
![image_1788773350548.png](./image_1788773350548.png "image_1788773350548.png")

![image_1788773359258.png](./image_1788773359258.png "image_1788773359258.png")